In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train_df = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test_df = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

In [ ]:
print('First 5 Rows Train Data Set')
train_df.head()

print('Train Data Set Information')
train_df.info()

In [ ]:
plt.figure(figsize=(10,6))

sns.histplot(train_df['SalePrice'], kde=True, bins=50)
plt.title('House Sale Price Distribution')
plt.xlabel('Sales Price')
plt.ylabel('Number of House')
plt.show()

In [ ]:
print('Sale Price Statistics')
print(train_df['SalePrice'].describe())

In [ ]:
print(f"Skewness Value: {train_df['SalePrice'].skew():.4f}")

In [ ]:
log_transformed_price = np.log1p(train_df['SalePrice'])

plt.figure(figsize=(10,6))
sns.histplot(log_transformed_price, kde=True, bins=50, color='green')
plt.title('Price Distribution After Logarithmic Transformation')
plt.xlabel('Log(SalePrice + 1')
plt.ylabel('Frequency')
plt.show()

print(f"\nPost-Transformation Distortion: {log_transformed_price.skew():.4f}")

In [ ]:
missing_data_count = train_df.isnull().sum()

total_rows = train_df.shape[0]
missing_data_percentage = (missing_data_count / total_rows)*100

missing_df = pd.concat([missing_data_count, missing_data_percentage], axis=1, keys=['Total Missing', 'Percentage'])

missing_df_filtered = missing_df[missing_df['Total Missing'] > 0].sort_values(by='Percentage', ascending=False)

print('Missing Values Report (High to Low)')
print(missing_df_filtered)

In [ ]:
categorical_features = train_df.select_dtypes(include=['object']).columns

print(f'Total OBJECT value: {len(categorical_features)}')

In [ ]:
numerical_features = train_df.select_dtypes(include=['int64', 'float64']).columns

numerical_features = numerical_features.drop(['Id', 'SalePrice'])

print(f'Total NUMERICAL value: {len(numerical_features)}')

In [ ]:
high_cardinality_features = []

for col in categorical_features:
    unique_count = train_df[col].nunique()

    if unique_count > 10:
        print(f"Row '{col}': {unique_count} is number of unique.")

print("\nPotentially problematic (high cardinality) columns:")
print(high_cardinality_features)

In [ ]:
train_ids = train_df['Id']
test_ids = test_df['Id']

train_df = train_df.drop('Id', axis=1)
test_df = test_df.drop('Id', axis=1)

y_train_log = np.log1p(train_df['SalePrice'])

train_df = train_df.drop('SalePrice', axis=1)

print(f'Before Concat Train Data Set Size: {train_df.shape}')
print(f'Before Concat Test Data Set Size: {test_df.shape}')

all_df = pd.concat([train_df, test_df]).reset_index(drop=True)

print(f'After Concat Data Set: {all_df.shape}')

### Imputation

In [ ]:
categorical_nan_as_none = [
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'MasVnrType'
]

In [ ]:
for col in categorical_nan_as_none:
    all_df[col].fillna("None", inplace=True)

print(f"The ‘NaN’ values in '{len(categorical_nan_as_none)} numerical columns were filled with ‘0’.")

In [ ]:
numerical_nan_as_zero = [
    'GarageYrBlt', 'GarageArea', 'GarageCars',
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
    'BsmtFullBath', 'BsmtHalfBath',
    'MasVnrArea'
]

In [ ]:
for col in numerical_nan_as_zero:
    all_df[col].fillna(0, inplace=True)

print(f"The ‘NaN’ values in '{len(numerical_nan_as_zero)}' numerical columns were filled with ‘0’.")

In [ ]:
missing_data_count_after = all_df.isnull().sum()
missing_data_percentage_after = (missing_data_count_after / all_df.shape[0]) * 100
missing_df_after = pd.concat([missing_data_count_after, missing_data_percentage_after], axis=1, keys=['Total Missing', 'Percentage'])
missing_df_filtered_after = missing_df_after[missing_df_after['Total Missing'] > 0].sort_values(by='Percentage', ascending=False)

print("POST-CLEANUP MISSING DATA REPORT")
print(missing_df_filtered_after)

In [ ]:
all_df['LotFrontage'] = all_df.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)

all_df['LotFrontage'].fillna(all_df['LotFrontage'].median(), inplace=True)

In [ ]:
all_df['MSZoning'].fillna(all_df['MSZoning'].mode()[0], inplace=True)
all_df['Utilities'].fillna(all_df['Utilities'].mode()[0], inplace=True)
all_df['Functional'].fillna(all_df['Functional'].mode()[0], inplace=True)
all_df['Electrical'].fillna(all_df['Electrical'].mode()[0], inplace=True)
all_df['Exterior1st'].fillna(all_df['Exterior1st'].mode()[0], inplace=True)
all_df['Exterior2nd'].fillna(all_df['Exterior2nd'].mode()[0], inplace=True)
all_df['KitchenQual'].fillna(all_df['KitchenQual'].mode()[0], inplace=True)
all_df['SaleType'].fillna(all_df['SaleType'].mode()[0], inplace=True)

In [ ]:
final_missing_check = all_df.isnull().sum().sum()

if final_missing_check == 0:
    print("FINAL CHECK SUCCESSFUL")
    print("There are NO MISSING (NaN) VALUES in the ‘all_df’ dataset.")
else:
    print(f"CONTROL FAILURE")
    print(f"There are still {final_missing_check} missing values. Please check the steps.")

In [ ]:
numeric_looking_cats = ['MSSubClass', 'OverallQual', 'OverallCond', 'YrSold', 'MoSold']

for col in numeric_looking_cats:
    all_df[col] = all_df[col].astype(str)

print(f"'{len(numeric_looking_cats)}' number-type columns were converted to the ‘string’ type.")

In [ ]:
all_df_encoded = pd.get_dummies(all_df, drop_first=True)

print(f"TRANSFORMATION COMPLETED")
print(f"Original Number of Columns (excluding Id/SalePrice): {all_df.shape[1]}")
print(f"Number of Columns After One-Hot Encoding: {all_df_encoded.shape[1]}")

print("The dataset is now 100% numerical and ready for modeling.")
all_df_encoded.head()

In [ ]:
X_train = all_df_encoded.iloc[:len(train_df)]
X_test = all_df_encoded.iloc[len(train_df):]

print(f"X_train (Features) Size: {X_train.shape}")
print(f"y_train_log (Target) Size: {y_train_log.shape}")
print(f"X_test (Estimated) Size: {X_test.shape}")

### Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

print(f"Original X_train: {X_train.iloc[0, :5].values}")
print(f"Scaled X_test: {X_train_scaled[0, :5]}")

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [ ]:
model_lr = LinearRegression()

model_lr.fit(X_train_scaled, y_train_log)

y_train_preds_lr = model_lr.predict(X_train_scaled)

rmse_lr = mean_squared_error(y_train_log, y_train_preds_lr)

print("Linear Regression")
print(f"Test Set RMSE: {rmse_lr:.4f}")

In [ ]:
from sklearn.model_selection import cross_val_score

In [ ]:
cv_scores_mse = cross_val_score(
    model_lr,
    X_train_scaled,
    y_train_log,
    cv=5,
    scoring='neg_mean_squared_error'
)
print("Cross-validation is complete.")

print(f"\nNegative MSE Scores for 5 Layers: {cv_scores_mse}")

mean_mse = -cv_scores_mse.mean()

mean_rmse = np.sqrt(mean_mse)

print(f"\nAverage Positive MSE: {mean_mse:.4f}")
print("--------------------------------------------------")
print(f"Reliable Average RMSE (Log Scale): {mean_rmse:.4f}")
print("--------------------------------------------------")

print(f"Training Set RMSE (Memorization Score): {rmse_lr:.4f}")
print(f"Cross-Validation RMSE (Actual Score): {mean_rmse:.4f}")

In [ ]:
from sklearn.linear_model import Ridge, Lasso

In [ ]:
model_ridge = Ridge(alpha=10)

cv_scores_mse_ridge = cross_val_score(
    model_ridge,
    X_train_scaled,
    y_train_log,
    cv=5,
    scoring='neg_mean_squared_error'
)
mean_rmse_ridge = np.sqrt(-cv_scores_mse_ridge.mean())
print(f"Ridge (alpha=10) Reliable Average RMSE: {mean_rmse_ridge:.4f}")

model_lasso = Lasso(alpha=0.0005)
cv_scores_mse_lasso = cross_val_score(
    model_lasso,
    X_train_scaled,
    y_train_log,
    cv=5,
    scoring='neg_mean_squared_error'
)
mean_rmse_lasso = np.sqrt(-cv_scores_mse_lasso.mean())
print(f"Lasso (alpha=0.0005) Reliable Average RMSE: {mean_rmse_lasso:.4f}")

In [ ]:
print("COMPARISON OF REGRESSION MODELS")
print(f"1. Linear Regression (Baseline) RMSE: {mean_rmse:.4f}")  
print(f"2. Ridge Regresyon (alpha=10) RMSE: {mean_rmse_ridge:.4f}")
print(f"3. Lasso Regresyon (alpha=0.0005) RMSE: {mean_rmse_lasso:.4f}")

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
model_rfr = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

cv_scores_mse_rfr = cross_val_score(
    model_rfr,
    X_train,  
    y_train_log,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1 
)

mean_rmse_rfr = np.sqrt(-cv_scores_mse_rfr.mean())
print(f"Random Forest Reliable Average RMSE: {mean_rmse_rfr:.4f}")

In [ ]:
print("\n--- TÜM MODELLER KARŞILAŞTIRMASI (CV RMSE) ---")
print(f"1. Linear Regression (Baseline) : {mean_rmse:.4f}")  
print(f"2. Ridge Regression (alpha=10)    : {mean_rmse_ridge:.4f}") 
print(f"3. Lasso Regression (alpha=0.0005) : {mean_rmse_lasso:.4f}")  
print(f"4. Random Forest (n=100)        : {mean_rmse_rfr:.4f}")

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
param_grid_rfr = {
    'n_estimators': [150, 250],      
    'max_depth': [15, 20, 25],         
    'min_samples_leaf': [1, 2, 4]      
}

model_rfr_base = RandomForestRegressor(random_state=42, n_jobs=-1)

grid_search_rfr = GridSearchCV(
    estimator=model_rfr_base,
    param_grid=param_grid_rfr,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=2 
)

print("Random Forest Hyperparameter Optimization (GridSearchCV) is starting...")
grid_search_rfr.fit(X_train, y_train_log)

print("\nOptimization is complete.")

In [ ]:
print(f"Best Hyperparameters (Random Forest): {grid_search_rfr.best_params_}")

best_neg_mse = grid_search_rfr.best_score_

best_rmse_rfr_tuned = np.sqrt(-best_neg_mse)

print(f"Optimized RFR Reliable Average RMSE: {best_rmse_rfr_tuned:.4f}")

In [ ]:
print("FINAL COMPARISON AFTER OPTIMIZATION")
print(f"Lasso Regresyon (Optimized) RMSE: {mean_rmse_lasso:.4f}")
print(f"Random Forest (Optimized) RMSE: {best_rmse_rfr_tuned:.4f}")

if best_rmse_rfr_tuned < mean_rmse_lasso:
    print("RESULT: Optimized Random Forest OUTPERFORMED Lasso. Our new best model!")
else:
    print("CONCLUSION: Optimized Lasso still yields a better score than Random Forest.")

In [ ]:
from xgboost import XGBRegressor

In [ ]:
model_xgb = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)

In [ ]:
cv_scores_mse_xgb = cross_val_score(
    model_xgb,
    X_train, 
    y_train_log,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

mean_rmse_xgb_default = np.sqrt(-cv_scores_mse_xgb.mean())
print(f"XGBoost (Default) Reliable Average RMSE: {mean_rmse_xgb_default:.4f}")

In [ ]:
print(f"1. Random Forest (Optimized) RMSE: {best_rmse_rfr_tuned:.4f}") 
print(f"2. XGBoost (Default Settings) RMSE: {mean_rmse_xgb_default:.4f}")

if mean_rmse_xgb_default < best_rmse_rfr_tuned:
    print("\nRESULT: XGBoost outperformed the optimized Random Forest. Our new best model!")
else:
    print("\nCONCLUSION: The optimized Random Forest appears to perform better than the default XGBoost.")

In [ ]:
final_model_xgb = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)
final_model_xgb.fit(X_train, y_train_log)
print("The training is complete.")

importances = final_model_xgb.feature_importances_

feature_importance_df = pd.Series(importances, index=X_train.columns)

top_n = 20
top_features = feature_importance_df.sort_values(ascending=False).head(top_n)

plt.figure(figsize=(10, 8))
sns.barplot(x=top_features.values, y=top_features.index)
plt.title(f'XGBoost - Most Important {top_n} Feature')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.show()

In [ ]:
test_predictions_log = final_model_xgb.predict(X_test)

test_predictions_dollar = np.expm1(test_predictions_log)

print("Forecasts have been successfully converted back to the dollar price.")

submission_df = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": test_predictions_dollar
})

submission_df.to_csv('submission_house_prices.csv', index=False)


print(submission_df.head())